## Try diagnosing the problem with the generator

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from generator import *
from sample_uniformly_per_color import *
import torch

/home/claireyy/miniconda3/envs/cse_556_invisibility_knit/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
tshirt_point, _ = sample_n_per_color("road.png", 6, 60)
tshirt_point = torch.from_numpy(tshirt_point)  # Convert numpy arrays to tensors


215
85
color 0 4215
color 1 3075
color 2 1751
color 3 1187
color 4 79
color 5 7968



(eog:447660): Gtk-WARNING **: 18:18:42.137: cannot open display: 


In [6]:
device = None
colors = torch.tensor([
            [194, 192, 172],
            [157, 178, 194],
            [106, 115, 84],
            [83, 100, 116],
            [41, 62, 85],
            [46, 114, 75]]).float().to(device)
colors = torch.div(colors, 255.)
num_colors = colors.shape[0]
fig_size_H = 340
fig_size_W = 860
resolution = 4
h, w = int(fig_size_H / resolution), int(fig_size_W / resolution)
coordinates = torch.stack(torch.meshgrid(torch.arange(h), torch.arange(w)), -1).to(device)
blur = 1
tau = 0.3
type = 'gumbel'

seeds_tshirt = torch.zeros(size=[h, w, num_colors], device=device).uniform_()

/home/claireyy/miniconda3/envs/cse_556_invisibility_knit/lib/python3.9/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /opt/conda/conda-bld/pytorch_1666643016022/work/aten/src/ATen/native/TensorShape.cpp:3190.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [7]:
k = 3
k2 = k * k
camouflage_kernel = nn.Conv2d(num_colors, num_colors, k, 1, int(k / 2)).to(device)
camouflage_kernel.weight.data.fill_(0)
camouflage_kernel.bias.data.fill_(0)
for i in range(num_colors):
    camouflage_kernel.weight[i, i, :, :].data.fill_(1 / k2)

In [8]:
print(tshirt_point.shape)
print(coordinates.shape)

torch.Size([6, 60, 2])
torch.Size([85, 215, 2])


In [9]:
prob_map = prob_fix_color(tshirt_point, coordinates, colors, h, w, blur).unsqueeze(0)
prob_map = camouflage_kernel(prob_map)
prob_map = prob_map.squeeze(0).permute(1, 2, 0)

gb_tshirt = -(-(seeds_tshirt + 1e-20).log() + 1e-20).log()

tex = gumbel_color_fix_seed(prob_map, gb_tshirt, colors, tau=tau, type=type)

tex_img = Image.fromarray(np.array(255*tex.squeeze(0).detach().cpu()).astype('uint8'))
tex_img.save('output_texture.png')